Dados estatísticos gerais

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# =====================================================================
# PREPARAÇÃO DOS DADOS
# =====================================================================
print("=" * 70)
print("ETAPA 0 — CARREGAMENTO E PREPARAÇÃO DOS DADOS")
print("=" * 70)

# Carregar os CSVs - ATUALIZADOS PARA OS COM CLIMA
try:
    df_br = pd.read_csv(
        "Campeonato_Brasileiro_Com_Clima.csv",
        encoding="utf-8"
    )
    df_pl = pd.read_csv(
        "Campeonato_Premier_League_Com_Clima.csv",
        encoding="utf-8"
    )
except FileNotFoundError as e:
    print(f"\n⚠ ERRO AO CARREGAR ARQUIVOS: {e}")
    exit()

# Criar coluna "Liga"
df_br["Liga"] = "Brasileirão Série A"
df_pl["Liga"] = "Premier League"

# Unir os dois DataFrames
df = pd.concat([df_br, df_pl], ignore_index=True)

# Converter colunas numéricas (tratamento de valores ausentes) - ADD COLUNAS DE CLIMA
colunas_numericas = [
    "Gols_Mandante", "Gols_Visitante",
    "Odd_Mandante", "Odd_Empate", "Odd_Visitante",
    "xG_Mandante", "xG_Visitante",
    "Finalizacoes_Mandante", "Finalizacoes_Visitante",
    "No_Alvo_Mandante", "No_Alvo_Visitante",
    "Defesas_Goleiro_Mandante", "Defesas_Goleiro_Visitante",
    "Chances_Claras_Mandante", "Chances_Claras_Visitante",
    "Escanteios_Mandante", "Escanteios_Visitante",
    "Amarelos_Mandante", "Amarelos_Visitante",
    "Vermelhos_Mandante", "Vermelhos_Visitante",
    "Trave_Mandante", "Trave_Visitante",
    "Faltas_Mandante", "Faltas_Visitante",
    "Temperatura_C", "Umidade_Relativa_%", "Velocidade_Vento_kmh"
]
for col in colunas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Público: limpar separadores de milhares e converter
if "Publico" in df.columns:
    df["Publico"] = (
        df["Publico"]
        .astype(str)
        .str.replace(r"[^\d]", "", regex=True)
    )
    df["Publico"] = pd.to_numeric(df["Publico"], errors="coerce")

# Criar coluna "Resultado"
def classificar_resultado(row):
    gm = row["Gols_Mandante"]
    gv = row["Gols_Visitante"]
    if pd.isna(gm) or pd.isna(gv):
        return np.nan
    if gm > gv:
        return "Mandante"
    elif gm == gv:
        return "Empate"
    else:
        return "Visitante"

df["Resultado"] = df.apply(classificar_resultado, axis=1)

# Converter Temporada para string para agrupamentos
df["Temporada"] = df["Temporada"].astype(str)

# Criar coluna Gols Totais e colunas de Frequência para a Seção 9
df["Gols_Total"] = df["Gols_Mandante"] + df["Gols_Visitante"]

df["Over_1_5"] = (df["Gols_Total"] > 1).astype(int)
df["Over_2_5"] = (df["Gols_Total"] > 2).astype(int)
df["Ambas_Marcam"] = ((df["Gols_Mandante"] > 0) & (df["Gols_Visitante"] > 0)).astype(int)

# Colunas binárias para os resultados (1X2)
df["Vit_Mandante"] = (df["Resultado"] == "Mandante").astype(int)
df["Empate_Res"] = (df["Resultado"] == "Empate").astype(int)
df["Vit_Visitante"] = (df["Resultado"] == "Visitante").astype(int)

print(f"  ✓ Brasileirão Série A: {len(df_br)} jogos carregados")
print(f"  ✓ Premier League:       {len(df_pl)} jogos carregados")
print(f"  ✓ Total combinado:      {len(df)} jogos")
print(f"  ✓ Temporadas BR: {sorted(df_br['Temporada'].unique())}")
print(f"  ✓ Temporadas PL: {sorted(df_pl['Temporada'].unique())}")
print()

# =====================================================================
# FUNÇÕES DE FORMATAÇÃO DE TABELA IMPRESSA (TIPO LaTeX)
# =====================================================================

def print_styled_season_stats_table(df_subset, league_name, caption, metric_name, decimal_places=3):
    base_cols = ["Temporada"]
    metric_cols = [f"{metric_name} Mandante", f"{metric_name} Visitante", f"{metric_name} Total"]
    col_headers = base_cols + metric_cols
    
    col_widths = [12, 18, 18, 18]
    if decimal_places > 0:
        col_formats = ["{:<12}"] + [f"{{:>18.{decimal_places}f}}" for _ in range(3)]
    else:
        col_formats = ["{:<12}"] + ["{:>18.0f}" for _ in range(3)]

    table_width = sum(col_widths) + 3 
    header_text = f"Liga: {league_name}"
    
    print(f"\n{caption}\n")
    print("=" * table_width)
    print(header_text)
    print("=" * table_width)
    
    header_row = ""
    for i, header in enumerate(col_headers):
        header_row += "{:<{w}} ".format(header, w=col_widths[i])
    print(header_row.strip())
    
    print("-" * table_width)
    
    for _, row in df_subset.iterrows():
        row_data = [
            row["Temporada"],
            row["Mandante"], 
            row["Visitante"],
            row["Total"]
        ]
        
        row_formatted = ""
        for i, data in enumerate(row_data):
            if isinstance(data, (float, int)):
                if metric_name in ["Moda", "Amplitude", "Máx"] and i > 0:
                     row_formatted += "{:>18.0f}".format(data) + " "
                else:
                    row_formatted += col_formats[i].format(data) + " "
            else:
                row_formatted += "{:<{w}} ".format(data, w=col_widths[i])
        print(row_formatted.strip())
        
    print("=" * table_width)
    footer_text = "Fonte: Flashscore"
    print(footer_text.center(table_width))
    print("\n")

def print_styled_frequency_table(df_subset, league_name, caption):
    col_headers = ["Temporada", "% Over 1.5 Gols", "% Over 2.5 Gols", "% Ambas Marcam"]
    col_widths = [12, 18, 18, 18]
    table_width = sum(col_widths) + 3

    print(f"\n{caption}\n")
    print("=" * table_width)
    print(f"Liga: {league_name}")
    print("=" * table_width)

    header_row = ""
    for i, header in enumerate(col_headers):
        header_row += "{:<{w}} ".format(header, w=col_widths[i])
    print(header_row.strip())

    print("-" * table_width)

    for _, row in df_subset.iterrows():
        r_season = row["Temporada"]
        r_o15 = f"{row['Over_1_5']:.1f}%"
        r_o25 = f"{row['Over_2_5']:.1f}%"
        r_btts = f"{row['Ambas_Marcam']:.1f}%"

        row_str = f"{r_season:<12} {r_o15:>18} {r_o25:>18} {r_btts:>18}"
        print(row_str)

    print("=" * table_width)
    print("Fonte: Flashscore".center(table_width))
    print("\n")

def print_styled_result_table(df_subset, league_name, caption):
    col_headers = ["Temporada", "% Vit. Mandante", "% Empate", "% Vit. Visitante"]
    col_widths = [12, 18, 18, 18]
    table_width = sum(col_widths) + 3

    print(f"\n{caption}\n")
    print("=" * table_width)
    print(f"Liga: {league_name}")
    print("=" * table_width)

    header_row = ""
    for i, header in enumerate(col_headers):
        header_row += "{:<{w}} ".format(header, w=col_widths[i])
    print(header_row.strip())

    print("-" * table_width)

    for _, row in df_subset.iterrows():
        r_season = row["Temporada"]
        r_m = f"{row['Vit_Mandante']:.1f}%"
        r_e = f"{row['Empate_Res']:.1f}%"
        r_v = f"{row['Vit_Visitante']:.1f}%"

        row_str = f"{r_season:<12} {r_m:>18} {r_e:>18} {r_v:>18}"
        print(row_str)

    print("=" * table_width)
    print("Fonte: Flashscore".center(table_width))
    print("\n")

# NOVA FUNÇÃO DE TABELA PARA O CLIMA
def print_styled_climate_table(df_subset, league_name, caption):
    col_headers = ["Temporada", "Temp. Média (°C)", "Umidade Média (%)", "Vento Médio (km/h)"]
    col_widths = [12, 18, 18, 18]
    table_width = sum(col_widths) + 3

    print(f"\n{caption}\n")
    print("=" * table_width)
    print(f"Liga: {league_name}")
    print("=" * table_width)

    header_row = ""
    for i, header in enumerate(col_headers):
        header_row += "{:<{w}} ".format(header, w=col_widths[i])
    print(header_row.strip())

    print("-" * table_width)

    for _, row in df_subset.iterrows():
        r_season = row["Temporada"]
        
        # Formata com 1 casa decimal, e caso seja NaN coloca um traço
        t = f"{row['Temperatura_C']:.1f}" if pd.notna(row['Temperatura_C']) else "-"
        u = f"{row['Umidade_Relativa_%']:.1f}" if pd.notna(row['Umidade_Relativa_%']) else "-"
        v = f"{row['Velocidade_Vento_kmh']:.1f}" if pd.notna(row['Velocidade_Vento_kmh']) else "-"

        row_str = f"{r_season:<12} {t:>18} {u:>18} {v:>18}"
        print(row_str)

    print("=" * table_width)
    print("Fonte: Open-Meteo & Flashscore".center(table_width))
    print("\n")

# =====================================================================
# FUNÇÕES MATEMÁTICAS CUSTOMIZADAS
# =====================================================================
def amplitude(x):
    if not x.dropna().empty:
        return x.max() - x.min()
    return np.nan

def moda(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else np.nan

amplitude.__name__ = 'Amplitude'
moda.__name__ = 'Moda'

# =====================================================================
# 1) QUANTIDADE DE JOGOS
# =====================================================================
print("=" * 70)
print("SEÇÃO 1 — QUANTIDADE DE JOGOS")
print("=" * 70)

jogos_por_liga = df.groupby("Liga").size().reset_index(name="Total_Jogos")
print("\n1a) Total de jogos por liga:")
print(jogos_por_liga.to_string(index=False))

jogos_liga_temp = (
    df.groupby(["Liga", "Temporada"])
    .size()
    .reset_index(name="Total_Jogos")
)
print("\n1b) Total de jogos por liga e temporada:")
print(jogos_liga_temp.to_string(index=False))

print(f"\n1c) Total geral (ambas ligas): {len(df)} jogos")
print()

# =====================================================================
# 2) MÉDIAS DE GOLS
# =====================================================================
print("=" * 70)
print("SEÇÃO 2 — MÉDIAS DE GOLS")
print("=" * 70)

media_gols_lt = (
    df.groupby(["Liga", "Temporada"])
    .agg(
        Media_Gols_Mandante=("Gols_Mandante", "mean"),
        Media_Gols_Visitante=("Gols_Visitante", "mean"),
        Media_Gols_Total=("Gols_Total", "mean") 
    )
    .reset_index()
)

print("\n2a) Média de gols por liga e temporada (FORMATO IMPRESSO):")

media_gols_br = media_gols_lt[media_gols_lt["Liga"] == "Brasileirão Série A"].sort_values("Temporada").copy()
media_gols_pl = media_gols_lt[media_gols_lt["Liga"] == "Premier League"].sort_values("Temporada").copy()

print_df_br = media_gols_br.rename(columns={
    "Media_Gols_Mandante": "Mandante",
    "Media_Gols_Visitante": "Visitante",
    "Media_Gols_Total": "Total"
})
print_styled_season_stats_table(
    print_df_br, 
    "Brasileirão Série A", 
    "Tabela 2.1: Média de gols por liga e temporada",
    "Média"
)

print_df_pl = media_gols_pl.rename(columns={
    "Media_Gols_Mandante": "Mandante",
    "Media_Gols_Visitante": "Visitante",
    "Media_Gols_Total": "Total"
})
print_styled_season_stats_table(
    print_df_pl, 
    "Premier League", 
    "Tabela 2.2: Média de gols por liga e temporada",
    "Média"
)

media_gols_liga = (
    df.groupby("Liga")
    .agg(
        Media_Gols_Mandante=("Gols_Mandante", "mean"),
        Media_Gols_Visitante=("Gols_Visitante", "mean"),
    )
    .reset_index()
)
media_gols_liga["Media_Gols_Mandante"] = media_gols_liga["Media_Gols_Mandante"].round(3)
media_gols_liga["Media_Gols_Visitante"] = media_gols_liga["Media_Gols_Visitante"].round(3)
media_gols_liga["Media_Total"] = (
    media_gols_liga["Media_Gols_Mandante"] + media_gols_liga["Media_Gols_Visitante"]
).round(3)

print("\n2b) Média geral de gols por liga:")
print(media_gols_liga.to_string(index=False))

media_m = df["Gols_Mandante"].mean()
media_v = df["Gols_Visitante"].mean()
print(f"\n2c) Média geral combinada (ambas ligas):")
print(f"    Mandante: {media_m:.3f}  |  Visitante: {media_v:.3f}  |  Total: {media_m + media_v:.3f}")
print()


# =====================================================================
# 3) RESULTADO DAS PARTIDAS
# =====================================================================
print("=" * 70)
print("SEÇÃO 3 — RESULTADO DAS PARTIDAS")
print("=" * 70)

df_res = df.dropna(subset=["Resultado"])

cont_lt = (
    df_res.groupby(["Liga", "Temporada", "Resultado"])
    .size()
    .reset_index(name="Contagem")
)
print("\n3a-i) Contagem de resultados por liga e temporada:")
print(cont_lt.to_string(index=False))

cont_liga = (
    df_res.groupby(["Liga", "Resultado"])
    .size()
    .reset_index(name="Contagem")
)
print("\n3a-ii) Contagem de resultados por liga:")
print(cont_liga.to_string(index=False))

cont_geral = (
    df_res.groupby("Resultado")
    .size()
    .reset_index(name="Contagem")
)
print("\n3a-iii) Contagem de resultados geral:")
print(cont_geral.to_string(index=False))

def calc_prob_observada(grupo):
    total = len(grupo)
    resultado_counts = grupo["Resultado"].value_counts()
    prob = {}
    for r in ["Mandante", "Empate", "Visitante"]:
        prob[f"Prob_Obs_{r}(%)"] = round(resultado_counts.get(r, 0) / total * 100, 2)
    return pd.Series(prob)

prob_obs_lt = df_res.groupby(["Liga", "Temporada"]).apply(
    calc_prob_observada, include_groups=False
).reset_index()
print("\n3b-i) Probabilidade observada por liga e temporada (%):")
print(prob_obs_lt.to_string(index=False))

prob_obs_liga = df_res.groupby("Liga").apply(
    calc_prob_observada, include_groups=False
).reset_index()
print("\n3b-ii) Probabilidade observada por liga (%):")
print(prob_obs_liga.to_string(index=False))

total_jogos = len(df_res)
r_counts = df_res["Resultado"].value_counts()
print("\n3b-iii) Probabilidade observada geral (%):")
for r in ["Mandante", "Empate", "Visitante"]:
    prob = r_counts.get(r, 0) / total_jogos * 100
    print(f"    {r}: {prob:.2f}%")
print()

# =====================================================================
# 4) COMPARAÇÃO COM AS ODDS
# =====================================================================
print("=" * 70)
print("SEÇÃO 4 — COMPARAÇÃO COM AS ODDS (PROBABILIDADES IMPLÍCITAS)")
print("=" * 70)

df_odds = df.dropna(subset=["Odd_Mandante", "Odd_Empate", "Odd_Visitante"]).copy()
df_odds = df_odds[
    (df_odds["Odd_Mandante"] > 0)
    & (df_odds["Odd_Empate"] > 0)
    & (df_odds["Odd_Visitante"] > 0)
]

df_odds["Prob_Impl_Mandante_Raw"] = 1.0 / df_odds["Odd_Mandante"]
df_odds["Prob_Impl_Empate_Raw"] = 1.0 / df_odds["Odd_Empate"]
df_odds["Prob_Impl_Visitante_Raw"] = 1.0 / df_odds["Odd_Visitante"]

df_odds["Soma_Probs"] = (
    df_odds["Prob_Impl_Mandante_Raw"]
    + df_odds["Prob_Impl_Empate_Raw"]
    + df_odds["Prob_Impl_Visitante_Raw"]
)

df_odds["Prob_Impl_Mandante"] = df_odds["Prob_Impl_Mandante_Raw"] / df_odds["Soma_Probs"]
df_odds["Prob_Impl_Empate"] = df_odds["Prob_Impl_Empate_Raw"] / df_odds["Soma_Probs"]
df_odds["Prob_Impl_Visitante"] = df_odds["Prob_Impl_Visitante_Raw"] / df_odds["Soma_Probs"]

def calc_prob_implicita_media(grupo):
    return pd.Series({
        "Prob_Impl_Mandante(%)": round(grupo["Prob_Impl_Mandante"].mean() * 100, 2),
        "Prob_Impl_Empate(%)": round(grupo["Prob_Impl_Empate"].mean() * 100, 2),
        "Prob_Impl_Visitante(%)": round(grupo["Prob_Impl_Visitante"].mean() * 100, 2),
        "Overround_Medio(%)": round((grupo["Soma_Probs"].mean() - 1) * 100, 2),
    })

prob_impl_lt = df_odds.groupby(["Liga", "Temporada"]).apply(
    calc_prob_implicita_media, include_groups=False
).reset_index()
print("\n4a) Probabilidade implícita média por liga e temporada (%):")
print(prob_impl_lt.to_string(index=False))

prob_impl_liga = df_odds.groupby("Liga").apply(
    calc_prob_implicita_media, include_groups=False
).reset_index()
print("\n4b) Probabilidade implícita média por liga (%):")
print(prob_impl_liga.to_string(index=False))

prob_impl_geral = calc_prob_implicita_media(df_odds)
print("\n4c) Probabilidade implícita média geral (%):")
for k, v in prob_impl_geral.items():
    print(f"    {k}: {v}%")

print("\n" + "-" * 50)
print("4d) TABELA COMPARATIVA — Observada vs Implícita (%)")
print("-" * 50)

def calc_prob_obs_subset(grupo):
    total = len(grupo)
    counts = grupo["Resultado"].value_counts()
    return pd.Series({
        "Prob_Obs_Mandante(%)": round(counts.get("Mandante", 0) / total * 100, 2),
        "Prob_Obs_Empate(%)": round(counts.get("Empate", 0) / total * 100, 2),
        "Prob_Obs_Visitante(%)": round(counts.get("Visitante", 0) / total * 100, 2),
    })

df_odds_res = df_odds.dropna(subset=["Resultado"])

comp_lt_obs = df_odds_res.groupby(["Liga", "Temporada"]).apply(
    calc_prob_obs_subset, include_groups=False
).reset_index()
comp_lt_impl = df_odds_res.groupby(["Liga", "Temporada"]).apply(
    calc_prob_implicita_media, include_groups=False
).reset_index()

comp_lt = comp_lt_obs.merge(comp_lt_impl, on=["Liga", "Temporada"])

for resultado in ["Mandante", "Empate", "Visitante"]:
    comp_lt[f"Dif_Abs_{resultado}(pp)"] = (
        comp_lt[f"Prob_Obs_{resultado}(%)"] - comp_lt[f"Prob_Impl_{resultado}(%)"]
    ).round(2)
    comp_lt[f"Dif_Rel_{resultado}(%)"] = (
        (comp_lt[f"Dif_Abs_{resultado}(pp)"] / comp_lt[f"Prob_Impl_{resultado}(%)"]) * 100
    ).round(2)

print("\n4d-i) Tabela comparativa por liga e temporada:")
cols_show = ["Liga", "Temporada"]
for r in ["Mandante", "Empate", "Visitante"]:
    cols_show.extend([f"Prob_Obs_{r}(%)", f"Prob_Impl_{r}(%)", f"Dif_Abs_{r}(pp)", f"Dif_Rel_{r}(%)"])
print(comp_lt[cols_show].to_string(index=False))

comp_liga_obs = df_odds_res.groupby("Liga").apply(
    calc_prob_obs_subset, include_groups=False
).reset_index()
comp_liga_impl = df_odds_res.groupby("Liga").apply(
    calc_prob_implicita_media, include_groups=False
).reset_index()
comp_liga = comp_liga_obs.merge(comp_liga_impl, on="Liga")
for resultado in ["Mandante", "Empate", "Visitante"]:
    comp_liga[f"Dif_Abs_{resultado}(pp)"] = (
        comp_liga[f"Prob_Obs_{resultado}(%)"] - comp_liga[f"Prob_Impl_{resultado}(%)"]
    ).round(2)
    comp_liga[f"Dif_Rel_{resultado}(%)"] = (
        (comp_liga[f"Dif_Abs_{resultado}(pp)"] / comp_liga[f"Prob_Impl_{resultado}(%)"]) * 100
    ).round(2)

print("\n4d-ii) Tabela comparativa por liga (total):")
cols_liga = ["Liga"]
for r in ["Mandante", "Empate", "Visitante"]:
    cols_liga.extend([f"Prob_Obs_{r}(%)", f"Prob_Impl_{r}(%)", f"Dif_Abs_{r}(pp)", f"Dif_Rel_{r}(%)"])
print(comp_liga[cols_liga].to_string(index=False))

total_c = len(df_odds_res)
counts_c = df_odds_res["Resultado"].value_counts()
obs_geral = {
    "Prob_Obs_Mandante": round(counts_c.get("Mandante", 0) / total_c * 100, 2),
    "Prob_Obs_Empate": round(counts_c.get("Empate", 0) / total_c * 100, 2),
    "Prob_Obs_Visitante": round(counts_c.get("Visitante", 0) / total_c * 100, 2),
}
impl_geral = {
    "Prob_Impl_Mandante": prob_impl_geral["Prob_Impl_Mandante(%)"],
    "Prob_Impl_Empate": prob_impl_geral["Prob_Impl_Empate(%)"],
    "Prob_Impl_Visitante": prob_impl_geral["Prob_Impl_Visitante(%)"],
}

print("\n4d-iii) Tabela comparativa geral (combinada):")
for r in ["Mandante", "Empate", "Visitante"]:
    obs_v = obs_geral[f"Prob_Obs_{r}"]
    impl_v = impl_geral[f"Prob_Impl_{r}"]
    dif_a = round(obs_v - impl_v, 2)
    dif_p = round((dif_a / impl_v) * 100, 2) if impl_v != 0 else 0
    print(f"    {r}: Obs={obs_v}%  Impl={impl_v}%  Dif_Abs={dif_a}pp  Dif_Rel={dif_p}%")

print()

# =====================================================================
# 5) PÚBLICO MÉDIO
# =====================================================================
print("=" * 70)
print("SEÇÃO 5 — PÚBLICO MÉDIO")
print("=" * 70)

df_pub = df[(df["Publico"].notna()) & (df["Publico"] > 0)]

pub_lt = (
    df_pub.groupby(["Liga", "Temporada"])
    .agg(Publico_Medio=("Publico", "mean"), Jogos_com_Publico=("Publico", "count"))
    .reset_index()
)
pub_lt["Publico_Medio"] = pub_lt["Publico_Medio"].round(0).astype(int)
print("\n5a) Público médio por liga e temporada:")
print(pub_lt.to_string(index=False))

pub_liga = (
    df_pub.groupby("Liga")
    .agg(Publico_Medio=("Publico", "mean"), Jogos_com_Publico=("Publico", "count"))
    .reset_index()
)
pub_liga["Publico_Medio"] = pub_liga["Publico_Medio"].round(0).astype(int)
print("\n5b) Público médio por liga:")
print(pub_liga.to_string(index=False))

if not df_pub.empty:
    pub_geral = df_pub["Publico"].mean()
    print(f"\n5c) Público médio geral: {pub_geral:,.0f}")
else:
    print("\n5c) Público médio geral: N/A (sem dados válidos de público)")
print()

# =====================================================================
# 6) MÉDIAS ADICIONAIS
# =====================================================================
print("=" * 70)
print("SEÇÃO 6 — MÉDIAS ADICIONAIS (Finalizações e Defesas do Goleiro)")
print("=" * 70)

colunas_adicionais = [
    "Defesas_Goleiro_Mandante",
    "Defesas_Goleiro_Visitante",
    "Finalizacoes_Mandante",
    "Finalizacoes_Visitante",
]

colunas_existentes = [c for c in colunas_adicionais if c in df.columns]

if colunas_existentes:
    medias_add_lt = df.groupby(["Liga", "Temporada"])[colunas_existentes].mean().reset_index()
    for c in colunas_existentes:
        medias_add_lt[c] = medias_add_lt[c].round(2)
    print("\n6a) Médias adicionais por liga e temporada:")
    print(medias_add_lt.to_string(index=False))

    medias_add_liga = df.groupby("Liga")[colunas_existentes].mean().reset_index()
    for c in colunas_existentes:
        medias_add_liga[c] = medias_add_liga[c].round(2)
    print("\n6b) Médias adicionais gerais por liga:")
    print(medias_add_liga.to_string(index=False))
else:
    print("\n  ⚠ Nenhuma das colunas adicionais foi encontrada nos dados.")

print()

# =====================================================================
# 7) ESTATÍSTICAS DESCRITIVAS AVANÇADAS
# =====================================================================
print("=" * 70)
print("SEÇÃO 7 — ESTATÍSTICAS DESCRITIVAS AVANÇADAS POR LIGA")
print("=" * 70)

cols_para_estatisticas = [
    "Gols_Mandante", "Gols_Visitante",
    "Finalizacoes_Mandante", "Finalizacoes_Visitante",
    "Faltas_Mandante", "Faltas_Visitante",
    "Amarelos_Mandante", "Amarelos_Visitante",
    "Publico"
]

cols_desc_existentes = [c for c in cols_para_estatisticas if c in df.columns]

if cols_desc_existentes:
    estatisticas_completas = df.groupby("Liga")[cols_desc_existentes].agg(
        ['mean', 'median', moda, 'std', 'max', 'min', amplitude]
    )
    
    estatisticas_completas.rename(columns={
        'mean': 'Média',
        'median': 'Mediana',
        'std': 'Desv_Padrão',
        'max': 'Máximo',
        'min': 'Mínimo'
    }, inplace=True)
    
    estatisticas_completas = estatisticas_completas.round(2)
    
    for col in cols_desc_existentes:
        print(f"\n--- Estatísticas para: {col.upper()} ---")
        print(estatisticas_completas[col].to_string())
else:
    print("\n  ⚠ Nenhuma coluna numérica encontrada para estatísticas avançadas.")

print()

# =====================================================================
# 8) RESUMO INTERPRETATIVO
# =====================================================================
print("=" * 70)
print("SEÇÃO 8 — RESUMO INTERPRETATIVO")
print("=" * 70)

print("""
PRINCIPAIS INSIGHTS ENCONTRADOS:

1. VOLUME DE JOGOS:
   Ambas as ligas apresentam volumes consistentes de partidas por temporada.

2. MÉDIAS DE GOLS (VER TABELAS DETALHADAS NO FINAL):
   O fator casa é um elemento importante em ambos os campeonatos.

3. DISTRIBUIÇÃO DE RESULTADOS:
   A probabilidade observada de vitória do mandante tende a ser maior.

4. ODDS VS REALIDADE:
   As diferenças entre probabilidades observadas e implícitas são geralmente pequenas.

5. PÚBLICO:
   Variações significativas notadas nas temporadas afetadas pela pandemia.
""")

print("=" * 70)
print("ANÁLISE INICIAL CONCLUÍDA COM SUCESSO!")
print("=" * 70)
print("\n" + "*" * 70)
print("INICIANDO GERAÇÃO DE TABELAS DETALHADAS POR TEMPORADA")
print("*" * 70)
print()


# =====================================================================
# 9) ESTATÍSTICAS DETALHADAS POR LIGA E TEMPORADA (FORMATO IMPRESSO)
# =====================================================================
print("\n" + "*" * 70)
print("INICIANDO GERAÇÃO DE TABELAS DETALHADAS POR TEMPORADA")
print("*" * 70)

print("=" * 70)
print("SEÇÃO 9 — ESTATÍSTICAS DETALHADAS POR LIGA E TEMPORADA (FORMATO IMPRESSO)")
print("=" * 70)

cols_para_estatisticas_detalhadas = [
    "Gols_Mandante", "Gols_Visitante", "Gols_Total"
]

stats_por_temporada_full = df.groupby(["Liga", "Temporada"])[cols_para_estatisticas_detalhadas].agg(
    [ 'mean', 'median', moda, amplitude, 'std', 'max' ]
)

stats_por_temporada_full.rename(columns={
    'mean': 'Média',
    'median': 'Mediana',
    'moda': 'Moda',
    'amplitude': 'Amplitude',
    'std': 'Desvio',
    'max': 'Máx'
}, inplace=True)

stats_por_temporada_flat = stats_por_temporada_full.copy()
stats_por_temporada_flat.columns = ['_'.join(col).strip() for col in stats_por_temporada_flat.columns.values]
stats_por_temporada_flat = stats_por_temporada_flat.reset_index()

freq_por_temporada = df.groupby(["Liga", "Temporada"])[["Over_1_5", "Over_2_5", "Ambas_Marcam"]].mean() * 100
freq_por_temporada = freq_por_temporada.reset_index()

freq_res_por_temporada = df.groupby(["Liga", "Temporada"])[["Vit_Mandante", "Empate_Res", "Vit_Visitante"]].mean() * 100
freq_res_por_temporada = freq_res_por_temporada.reset_index()

# AGRUPAMENTO PARA O CLIMA (NOVO)
clima_por_temporada = df.groupby(["Liga", "Temporada"])[["Temperatura_C", "Umidade_Relativa_%", "Velocidade_Vento_kmh"]].mean().reset_index()

metricas_para_imprimir = [
    ("Média", 3, "Média"), 
    ("Mediana", 3, "Mediana"), 
    ("Moda", 0, "Moda"), 
    ("Amplitude", 0, "Amplitude"),
    ("Desvio", 3, "Desvio Padrão"), 
    ("Máx", 0, "Máximos e Mínimos") 
]

ligas_presentes = df["Liga"].unique()

for liga in ligas_presentes:
    liga_subset = stats_por_temporada_flat[stats_por_temporada_flat["Liga"] == liga].sort_values("Temporada").copy()
    
    print(f"\n" + "*" * 50)
    print(f"TABELAS DETALHADAS PARA A LIGA: {liga.upper()}")
    print("*" * 50)
    
    # Imprime as tabelas contínuas de 9.1 até 9.6
    for metric_key, decimal_p, metric_display_name in metricas_para_imprimir:
        
        print_df = pd.DataFrame()
        print_df["Temporada"] = liga_subset["Temporada"]
        print_df["Mandante"] = liga_subset[f"Gols_Mandante_{metric_key}"]
        print_df["Visitante"] = liga_subset[f"Gols_Visitante_{metric_key}"]
        print_df["Total"] = liga_subset[f"Gols_Total_{metric_key}"]
        
        legenda_pref = "Tabela 9."
        tabela_id = metricas_para_imprimir.index((metric_key, decimal_p, metric_display_name)) + 1
        
        if metric_key == "Desvio":
            col_name_display = "Desvio"
        elif metric_key == "Máx":
            col_name_display = "Máx"
        else:
            col_name_display = metric_display_name
            
        legenda = f"{legenda_pref}{tabela_id}: {metric_display_name} de gols por liga e temporada"
        print_styled_season_stats_table(print_df, liga, legenda, col_name_display, decimal_places=decimal_p)
    
    # Tabela 9.7
    freq_subset = freq_por_temporada[freq_por_temporada["Liga"] == liga].sort_values("Temporada")
    legenda_97 = "Tabela 9.7: Frequência de Gols (Over 2.5 e Ambas Marcam)"
    print_styled_frequency_table(freq_subset, liga, legenda_97)

    # Tabela 9.8
    res_subset = freq_res_por_temporada[freq_res_por_temporada["Liga"] == liga].sort_values("Temporada")
    legenda_98 = "Tabela 9.8: Frequência de Resultados (1X2)"
    print_styled_result_table(res_subset, liga, legenda_98)

    # ---------------------------------------------------------------------
    # Nova Tabela 9.9: Médias Climáticas
    # ---------------------------------------------------------------------
    clima_subset = clima_por_temporada[clima_por_temporada["Liga"] == liga].sort_values("Temporada")
    legenda_99 = "Tabela 9.9: Condições Climáticas Médias por Temporada"
    print_styled_climate_table(clima_subset, liga, legenda_99)

print()
print("=" * 70)
print("TODAS AS TABELAS DETALHADAS FORAM GERADAS COM SUCESSO!")
print("=" * 70)

ETAPA 0 — CARREGAMENTO E PREPARAÇÃO DOS DADOS
  ✓ Brasileirão Série A: 2346 jogos carregados
  ✓ Premier League:       2583 jogos carregados
  ✓ Total combinado:      4929 jogos
  ✓ Temporadas BR: [2020, 2021, 2022, 2023, 2024, 2025, 2026]
  ✓ Temporadas PL: [2019, 2020, 2021, 2022, 2023, 2024, 2025]

SEÇÃO 1 — QUANTIDADE DE JOGOS

1a) Total de jogos por liga:
               Liga  Total_Jogos
Brasileirão Série A         2346
     Premier League         2583

1b) Total de jogos por liga e temporada:
               Liga Temporada  Total_Jogos
Brasileirão Série A      2020          380
Brasileirão Série A      2021          380
Brasileirão Série A      2022          380
Brasileirão Série A      2023          380
Brasileirão Série A      2024          380
Brasileirão Série A      2025          380
Brasileirão Série A      2026           66
     Premier League      2019          380
     Premier League      2020          380
     Premier League      2021          380
     Premier League    